In [ ]:
import preprocessing
from multitensor_systems import multify
from multitensor_systems_vec import flat_multitensor, unpack_flat
import torch

def create_mts(channel_dim=16, n_tasks=5):
    # create real sized multitensor for testing
    task_nums = list(range(n_tasks))
    split = "training"  # "training", "evaluation, or "test"
    tasks = preprocessing.preprocess_tasks(split, task_nums)
    MTs = []

    @multify
    def init(dims, _, multitensor_system, channel_dim):
        shape = multitensor_system.shape(dims, channel_dim)
        mean = torch.randn(shape)
        mean.requires_grad=True
        return mean

    for task in tasks:
        multitensor_system = task.multitensor_system
        mt = init(multitensor_system.make_multitensor(channel_dim), multitensor_system, channel_dim)
        MTs.append(mt)
    return MTs

@multify
def get_grads(dims, mt):
    return mt.grad

def abs_diff(a, b):
    return torch.abs(a - b).mean()

MTs = create_mts()
def test_meta(MTs, fn, fn_vec):
    FTs = [flat_multitensor(mt, debug=True) for mt in MTs]
    for mt, ft in zip(MTs, FTs):
        # forward pass
        mt2 = fn(mt)
        ft2 = fn_vec(ft)
        assert abs_diff(flat_multitensor(mt2).data, ft2.data) < 1e-4, f"forward pass failed"
        # backward pass
        flat_multitensor(mt2).data.sum().backward()
        ft2.data.sum().backward()
        assert abs_diff(flat_multitensor(get_grads(mt)).data, ft.data.grad) < 1e-3, f"backward pass failed"
    
    for mt in MTs:
        mt2 = fn(mt)
        flat_multitensor(mt2).data.sum().backward()

In [ ]:
import layers_vec
import layers
test_meta(MTs, layers.normalize, layers_vec.normalize)